## HuggingFace Gender Classification

Link: https://huggingface.co/rizvandwiki/gender-classification

In [5]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


# ============================================================================
# MODEL 1: HuggingFace ViT Gender Classifier (GPU + Batch Optimized)
# ============================================================================
class HuggingFaceGenderClassifier:
    def __init__(self, model_name="rizvandwiki/gender-classification"):
        from transformers import AutoImageProcessor, AutoModelForImageClassification
        
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading {model_name} on {self.device}...")
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name)
        self.model = self.model.to(self.device)
        self.model.eval()
        print("✓ Loaded successfully.")

    def predict_batch(self, image_paths):
        """
        image_paths: list of paths
        Returns: list of (gender, confidence)
        """

        # Load images
        images = []
        for p in image_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                images.append(None)

        # Remove None values (missing images)
        valid_indices = [i for i, img in enumerate(images) if img is not None]
        valid_images = [images[i] for i in valid_indices]

        if len(valid_images) == 0:
            return [(None, None) for _ in image_paths]

        # Preprocess batch
        inputs = self.processor(images=valid_images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()

        # Map predictions back
        results = [(None, None)] * len(image_paths)
        for j, i in enumerate(valid_indices):
            p = probs[j]
            cls = p.argmax()
            conf = float(p[cls])
            gender_raw = self.model.config.id2label[cls]
            gender = gender_raw.capitalize().replace("_", " ").replace("portrait", "").strip()
            results[i] = (gender, conf)

        return results


# ============================================================================
# EVALUATION FUNCTION (SIMPLE, FULLY SELF-CONTAINED)
# ============================================================================
def evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image"):

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )

    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = HuggingFaceGenderClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []

    # Collect image paths
    image_paths = []
    true_labels = []
    tones = []

    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row[image_type])

        if not os.path.exists(img):
            img = os.path.join(image_root, row[image_type])

        if os.path.exists(img):
            image_paths.append(img)
            true_labels.append(row["gender"].lower().strip())
            tones.append(int(row["mst_label"]))

    # Batch inference
    print("\nRunning batched inference...")
    predictions = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_preds = model.predict_batch(batch_paths)
        predictions.extend(batch_preds)

    # Build results
    for (pred_gender, conf), true_g, t in zip(predictions, true_labels, tones):
        if pred_gender is None:
            continue
        all_true.append(true_g)
        all_pred.append(pred_gender.lower())
        all_conf.append(conf)
        all_tone.append(t)

    # Save detailed results
    result_df = pd.DataFrame({
        "image_path": image_paths[:len(all_true)],
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone
    })
    result_df.to_csv(output_csv, index=False)
    print(f"\nSaved predictions → {output_csv}")

    # Metrics
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, c):
        idx = [i for i, v in enumerate(y) if v == c]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == c for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male") and (p=="male") for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male") for t,p in zip(y,yp))
        FN = sum((t=="male") and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }

    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    unique_tones = sorted(set(all_tone))
    for tone in unique_tones:
        idx = [i for i,t in enumerate(all_tone) if t==tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, result_df


### Casual Conversation v2

In [20]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\hf_gender_eval.csv"

hf_ccv2_faceonly_metrics, hf_ccv2_faceonly_per_tone, hf_ccv2_faceonly_result_df = evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")
print(hf_ccv2_faceonly_metrics)
print(hf_ccv2_faceonly_per_tone)
print(hf_ccv2_faceonly_result_df.head())

Loading rizvandwiki/gender-classification on cuda...
✓ Loaded successfully.

Running batched inference...


100%|██████████| 5559/5559 [42:47<00:00,  2.16it/s]



Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\hf_gender_eval.csv
{'overall_accuracy': 0.7152415061030118, 'male_accuracy': 0.962622028084782, 'female_accuracy': 0.5613137421220163, 'TP': 65672, 'TN': 61543, 'FP': 48098, 'FN': 2550}
{1: {'N': 1190, 'accuracy': 0.6579831932773109, 'male_accuracy': 0.9901315789473685, 'female_accuracy': 0.5440180586907449}, 2: {'N': 13337, 'accuracy': 0.7027067556421984, 'male_accuracy': 0.9744773282650013, 'female_accuracy': 0.5990263103376838}, 3: {'N': 33732, 'accuracy': 0.7031898494011621, 'male_accuracy': 0.9755419144446575, 'female_accuracy': 0.5813524414313911}, 4: {'N': 38273, 'accuracy': 0.6828834948919604, 'male_accuracy': 0.9690762732127202, 'female_accuracy': 0.5433024955298142}, 5: {'N': 55797, 'accuracy': 0.7603096940695736, 'male_accuracy': 0.9629658875552748, 'female_accuracy': 0.5918474515080901}, 6: {'N': 23074, 'accuracy': 0.7139204299211234, 'male_accuracy': 0.937828883258624, 'female_accuracy': 0.534576959

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Images"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\hf_gender_eval.csv"

hf_ccv2_metrics, hf_ccv2_per_tone, hf_ccv2_result_df = evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32, image_type="original_image")
print(hf_ccv2_metrics)
print(hf_ccv2_per_tone)
print(hf_ccv2_result_df.head())

Loading rizvandwiki/gender-classification on cuda...
✓ Loaded successfully.

Running batched inference...


100%|██████████| 5559/5559 [1:55:13<00:00,  1.24s/it]  



Saved predictions → G:\Thesis\CasualConversationv2_Dataset\Images\hf_gender_eval.csv
{'overall_accuracy': 0.831280255027746, 'male_accuracy': 0.9869543549001788, 'female_accuracy': 0.7344150454665682, 'TP': 67332, 'TN': 80522, 'FP': 29119, 'FN': 890}
{1: {'N': 1190, 'accuracy': 0.7495798319327731, 'male_accuracy': 1.0, 'female_accuracy': 0.6636568848758465}, 2: {'N': 13337, 'accuracy': 0.8185498987778361, 'male_accuracy': 0.9942981265272876, 'female_accuracy': 0.7515019680961259}, 3: {'N': 33732, 'accuracy': 0.8425827107790822, 'male_accuracy': 0.9835986955687703, 'female_accuracy': 0.7794988415000429}, 4: {'N': 38273, 'accuracy': 0.830742298748465, 'male_accuracy': 0.9772057065433968, 'female_accuracy': 0.759309647827101}, 5: {'N': 55797, 'accuracy': 0.8408337365808197, 'male_accuracy': 0.9917482627921668, 'female_accuracy': 0.7153828481407332}, 6: {'N': 23074, 'accuracy': 0.8366126376007628, 'male_accuracy': 0.982069771974274, 'female_accuracy': 0.7201061504839213}, 7: {'N': 6438, '

### FACET

In [ ]:
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\hf_gender_eval.csv"

hf_facet_faceonly_metrics, hf_facet_faceonly_per_tone, hf_facet_faceonly_result_df = evaluate_huggingface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")
print(hf_facet_faceonly_metrics)
print(hf_facet_faceonly_per_tone)
print(hf_facet_faceonly_result_df.head())

Loading rizvandwiki/gender-classification on cuda...
✓ Loaded successfully.

Running batched inference...


100%|██████████| 84/84 [00:41<00:00,  2.00it/s]


Saved predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\hf_gender_eval.csv
{'overall_accuracy': 0.7889428464699291, 'male_accuracy': 0.876633559853633, 'female_accuracy': 0.569371727748691, 'TP': 1677, 'TN': 435, 'FP': 329, 'FN': 236}
{1: {'N': 70, 'accuracy': 0.7857142857142857, 'male_accuracy': 0.7441860465116279, 'female_accuracy': 0.8518518518518519}, 2: {'N': 559, 'accuracy': 0.817531305903399, 'male_accuracy': 0.8858560794044665, 'female_accuracy': 0.6410256410256411}, 3: {'N': 693, 'accuracy': 0.7705627705627706, 'male_accuracy': 0.8605150214592274, 'female_accuracy': 0.5859030837004405}, 4: {'N': 485, 'accuracy': 0.8041237113402062, 'male_accuracy': 0.8842105263157894, 'female_accuracy': 0.5142857142857142}, 5: {'N': 349, 'accuracy': 0.7765042979942693, 'male_accuracy': 0.8831168831168831, 'female_accuracy': 0.5677966101694916}, 6: {'N': 288, 'accuracy': 0.7951388888888888, 'male_accuracy': 0.9018691588785047, 'female_accuracy': 0.4864864864864865}, 7: {'N'

## Realistic Gender Classifier

Link: https://huggingface.co/prithivMLmods/Realistic-Gender-Classification

In [9]:
import os
import torch
import pandas as pd
import numpy as np
from PIL import Image
from tqdm import tqdm


# ============================================================================
# MODEL 2: Realistic Gender Classification (SigLIP-based, GPU + Batching)
# ============================================================================
class RealisticGenderClassifier:
    """
    prithivMLmods/Realistic-Gender-Classification
    SigLIP backbone — very strong performance
    """

    def __init__(self):
        from transformers import AutoImageProcessor, AutoModelForImageClassification
        
        model_name = "prithivMLmods/Realistic-Gender-Classification"
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

        print(f"Loading {model_name} on {self.device}...")
        self.processor = AutoImageProcessor.from_pretrained(model_name)
        self.model = AutoModelForImageClassification.from_pretrained(model_name)
        self.model = self.model.to(self.device)
        self.model.eval()
        print("✓ Model loaded successfully.")

    def predict_batch(self, image_paths):
        """
        image_paths: list[str]
        Returns: list of (gender, confidence)
        """

        # Load images
        images = []
        for p in image_paths:
            try:
                img = Image.open(p).convert("RGB")
                images.append(img)
            except:
                images.append(None)

        valid_idx = [i for i, img in enumerate(images) if img is not None]
        valid_images = [images[i] for i in valid_idx]

        if len(valid_images) == 0:
            return [(None, None) for _ in image_paths]

        # Preprocess
        inputs = self.processor(images=valid_images, return_tensors="pt")
        inputs = {k: v.to(self.device) for k, v in inputs.items()}

        # Forward pass
        with torch.no_grad():
            outputs = self.model(**inputs)
            probs = torch.nn.functional.softmax(outputs.logits, dim=-1).cpu().numpy()

        # Map predictions
        results = [(None, None)] * len(image_paths)
        for j, i in enumerate(valid_idx):
            p = probs[j]
            cls = p.argmax()
            conf = float(p[cls])

            # Label format: "male portrait" → take first token
            label = self.model.config.id2label[cls]
            gender = label.split()[0].capitalize()

            results[i] = (gender, conf)

        return results


# ============================================================================
# EVALUATION FUNCTION (SELF-CONTAINED)
# ============================================================================
def evaluate_realistic(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image"):

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )
    
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = RealisticGenderClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []

    # Collect paths
    image_paths = []
    true_labels = []
    tones = []

    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row[image_type])

        if not os.path.exists(img):
            img = os.path.join(image_root, row[image_type])

        if os.path.exists(img):
            image_paths.append(img)
            true_labels.append(row["gender"].lower().strip())
            tones.append(int(row["mst_label"]))

    # Batch inference
    print("\nRunning batched inference (Realistic Gender Model)...")
    predictions = []

    for i in tqdm(range(0, len(image_paths), batch_size)):
        batch_paths = image_paths[i:i+batch_size]
        batch_preds = model.predict_batch(batch_paths)
        predictions.extend(batch_preds)

    # Build results
    for (pred_g, conf), tg, t in zip(predictions, true_labels, tones):
        if pred_g is None:
            continue
        all_true.append(tg)
        all_pred.append(pred_g.lower())
        all_conf.append(conf)
        all_tone.append(t)

    # Save results CSV
    out_df = pd.DataFrame({
        "image_path": image_paths[:len(all_true)],
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone
    })
    out_df.to_csv(output_csv, index=False)
    print(f"\nSaved Realistic Model predictions → {output_csv}")

    # Metrics
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, c):
        idx = [i for i, v in enumerate(y) if v == c]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == c for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }

    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i,t in enumerate(all_tone) if t==tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df


### Casual Conversation v2

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\realistic_gender_eval.csv"

realistic_ccv2_faceonly_metrics, realistic_ccv2_faceonly_per_tone, realistic_ccv2_faceonly_result_df = evaluate_realistic(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")

print(realistic_ccv2_faceonly_metrics)
print(realistic_ccv2_faceonly_per_tone)
print(realistic_ccv2_faceonly_result_df.head())

Loading prithivMLmods/Realistic-Gender-Classification on cuda...
✓ Model loaded successfully.

Running batched inference (Realistic Gender Model)...


100%|██████████| 5559/5559 [22:30<00:00,  4.12it/s]



Saved Realistic Model predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\realistic_gender_eval.csv
{'overall_accuracy': 0.885810989356977, 'male_accuracy': 0.9242473102518249, 'female_accuracy': 0.8618947291615363, 'TP': 63054, 'TN': 94499, 'FP': 15142, 'FN': 5168}
{1: {'N': 1190, 'accuracy': 0.7563025210084033, 'male_accuracy': 0.9769736842105263, 'female_accuracy': 0.6805869074492099}, 2: {'N': 13337, 'accuracy': 0.8702106920596836, 'male_accuracy': 0.9679609014390442, 'female_accuracy': 0.8329189973068158}, 3: {'N': 33732, 'accuracy': 0.8933060595280445, 'male_accuracy': 0.9458085555342414, 'female_accuracy': 0.869818930747447}, 4: {'N': 38273, 'accuracy': 0.8932406657434745, 'male_accuracy': 0.9394277516537818, 'female_accuracy': 0.8707144523050611}, 5: {'N': 55797, 'accuracy': 0.9056400881767837, 'male_accuracy': 0.9182722678458622, 'female_accuracy': 0.8951393219337688}, 6: {'N': 23074, 'accuracy': 0.8793880558204039, 'male_accuracy': 0.8843305398557786, 'female

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Images"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\realistic_gender_eval.csv"

realistic_ccv2_metrics, realistic_ccv2_per_tone, realistic_ccv2_result_df = evaluate_realistic(csv_path, image_root, output_csv, batch_size=32, image_type="original_image")

print(realistic_ccv2_metrics)
print(realistic_ccv2_per_tone)
print(realistic_ccv2_result_df.head())

Loading prithivMLmods/Realistic-Gender-Classification on cuda...
✓ Model loaded successfully.

Running batched inference (Realistic Gender Model)...


100%|██████████| 5559/5559 [1:48:55<00:00,  1.18s/it]  



Saved Realistic Model predictions → G:\Thesis\CasualConversationv2_Dataset\Images\realistic_gender_eval.csv
{'overall_accuracy': 0.8159201182932931, 'male_accuracy': 0.9889331887074551, 'female_accuracy': 0.7082660683503434, 'TP': 67467, 'TN': 77655, 'FP': 31986, 'FN': 755}
{1: {'N': 1190, 'accuracy': 0.7974789915966387, 'male_accuracy': 1.0, 'female_accuracy': 0.7279909706546276}, 2: {'N': 13337, 'accuracy': 0.7920821774012147, 'male_accuracy': 0.9934835731740429, 'female_accuracy': 0.7152475657758443}, 3: {'N': 33732, 'accuracy': 0.8238467923633345, 'male_accuracy': 0.9898331095338576, 'female_accuracy': 0.7495923796447267}, 4: {'N': 38273, 'accuracy': 0.794528780079952, 'male_accuracy': 0.9860524428150156, 'female_accuracy': 0.7011194900101065}, 5: {'N': 55797, 'accuracy': 0.8462820581751707, 'male_accuracy': 0.98981364497789, 'female_accuracy': 0.7269683941054843}, 6: {'N': 23074, 'accuracy': 0.8198405131316634, 'male_accuracy': 0.9824595595400507, 'female_accuracy': 0.68958788635

### FACET

In [10]:
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\realistic_gender_eval.csv"

realistic_facet_faceonly_metrics, realistic_facet_faceonly_per_tone, realistic_facet_faceonly_result_df = evaluate_realistic(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")
print(realistic_facet_faceonly_metrics)
print(realistic_facet_faceonly_per_tone)
print(realistic_facet_faceonly_result_df.head())

Loading prithivMLmods/Realistic-Gender-Classification on cuda...
✓ Model loaded successfully.

Running batched inference (Realistic Gender Model)...


100%|██████████| 84/84 [00:23<00:00,  3.64it/s]


Saved Realistic Model predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\realistic_gender_eval.csv
{'overall_accuracy': 0.8233096750093388, 'male_accuracy': 0.8745426032409828, 'female_accuracy': 0.6950261780104712, 'TP': 1673, 'TN': 531, 'FP': 233, 'FN': 240}
{1: {'N': 70, 'accuracy': 0.7714285714285715, 'male_accuracy': 0.7906976744186046, 'female_accuracy': 0.7407407407407407}, 2: {'N': 559, 'accuracy': 0.8354203935599285, 'male_accuracy': 0.8982630272952854, 'female_accuracy': 0.6730769230769231}, 3: {'N': 693, 'accuracy': 0.8311688311688312, 'male_accuracy': 0.871244635193133, 'female_accuracy': 0.748898678414097}, 4: {'N': 485, 'accuracy': 0.8247422680412371, 'male_accuracy': 0.8789473684210526, 'female_accuracy': 0.6285714285714286}, 5: {'N': 349, 'accuracy': 0.7650429799426934, 'male_accuracy': 0.7835497835497836, 'female_accuracy': 0.7288135593220338}, 6: {'N': 288, 'accuracy': 0.8819444444444444, 'male_accuracy': 0.9252336448598131, 'femal

## DeepFace

Link: https://github.com/serengil/deepface

In [25]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm


# ============================================================================
# DEEPFACE MODEL (NO BATCHING — DEEPFACE DOES NOT SUPPORT BATCH INPUT)
# ============================================================================
class DeepFaceClassifier:
    """
    DeepFace gender classifier
    Requires:
        pip install deepface tensorflow
    Runs CPU-only unless your TensorFlow is GPU-enabled.
    """

    def __init__(self):
        try:
            from deepface import DeepFace
            self.deepface = DeepFace
            print("✓ DeepFace loaded successfully.")
        except ImportError:
            print("✗ DeepFace not installed. Run: pip install deepface tensorflow")
            self.deepface = None

    def predict_single(self, image_path):
        if self.deepface is None:
            return None, None

        try:
            result = self.deepface.analyze(
                img_path=image_path,
                actions=['gender'],
                detector_backend='opencv',
                enforce_detection=False,
                silent=True
            )

            # DeepFace sometimes returns list of results
            if isinstance(result, list):
                result = result[0]

            gender = result['dominant_gender'].capitalize()
            confidence = float(result['gender'][result['dominant_gender']])

            return gender, confidence

        except Exception:
            return None, None

def normalize_gender(g):
    if g is None:
        return None
    g = g.lower().strip()
    if g == "man":
        return "male"
    if g == "woman":
        return "female"
    return g

# ============================================================================
# EVALUATION FUNCTION (SELF-CONTAINED)
# ============================================================================
def evaluate_deepface(csv_path, image_root, output_csv, image_type="cropped_image"):

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )
    
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = DeepFaceClassifier()

    all_true = []
    all_pred = []
    all_conf = []
    all_tone = []
    image_paths = []

    print("\nCollecting image paths...")
    for _, row in df.iterrows():
        img = os.path.join(image_root, row["subject_id"], row[image_type])

        if not os.path.exists(img):
            img = os.path.join(image_root, row[image_type])

        if os.path.exists(img):
            image_paths.append(img)
            all_true.append(row["gender"].lower().strip())
            all_tone.append(int(row["mst_label"]))

    print(f"Found {len(image_paths)} images to process.")

    # Run DeepFace sequentially
    print("\nRunning DeepFace inference...")
    predictions = []
    for img in tqdm(image_paths):
        pred_gender, conf = model.predict_single(img)
        pred_gender = normalize_gender(pred_gender)
        predictions.append((pred_gender, conf))

    # Filter out invalid predictions
    final_true = []
    final_pred = []
    final_conf = []
    final_tone = []
    final_paths = []

    for (pg, cf), tg, t, p in zip(predictions, all_true, all_tone, image_paths):
        if pg is None:
            continue

        final_true.append(tg)
        final_pred.append(pg.lower())
        final_conf.append(cf)
        final_tone.append(t)
        final_paths.append(p)

    # Save results to CSV
    result_df = pd.DataFrame({
        "image_path": final_paths,
        "true_gender": final_true,
        "pred_gender": final_pred,
        "confidence": final_conf,
        "mst_label": final_tone
    })
    result_df.to_csv(output_csv, index=False)
    print(f"\nSaved DeepFace predictions → {output_csv}")

    # ----------------------------------------
    # METRICS
    # ----------------------------------------
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))

    def acc_class(y, yp, c):
        idx = [i for i,v in enumerate(y) if v == c]
        if len(idx) == 0:
            return 0.0
        return np.mean([yp[j] == c for j in idx])

    def confusion(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(final_true, final_pred),
        "male_accuracy": acc_class(final_true, final_pred, "male"),
        "female_accuracy": acc_class(final_true, final_pred, "female"),
    }

    TP, TN, FP, FN = confusion(final_true, final_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone accuracy
    per_tone = {}
    for tone in sorted(set(final_tone)):
        idx = [i for i,t in enumerate(final_tone) if t == tone]
        yt = [final_true[j] for j in idx]
        yp = [final_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, result_df


### Casual Conversation v2

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\deepface_gender_eval.csv"

deepface_ccv2_faceonly_metrics, deepface_ccv2_faceonly_per_tone, deepface_ccv2_faceonly_result_df = evaluate_deepface(csv_path, image_root, output_csv, image_type="cropped_image")

print(deepface_ccv2_faceonly_metrics)
print(deepface_ccv2_faceonly_per_tone)
print(deepface_ccv2_faceonly_result_df.head())

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Images"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\deepface_gender_eval.csv"

deepface_ccv2_metrics, deepface_ccv2_per_tone, deepface_ccv2_result_df = evaluate_deepface(csv_path, image_root, output_csv, image_type="original_image")

print(deepface_ccv2_metrics)
print(deepface_ccv2_per_tone)
print(deepface_ccv2_result_df.head())

### FACET

In [26]:
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\deepface_gender_eval.csv"

deepface_facet_faceonly_metrics, deepface_facet_faceonly_per_tone, deepface_facet_faceonly_result_df = evaluate_deepface(csv_path, image_root, output_csv, image_type="cropped_image")
print(deepface_facet_faceonly_metrics)
print(deepface_facet_faceonly_per_tone)
print(deepface_facet_faceonly_result_df.head())

✓ DeepFace loaded successfully.

Found 2677 images to process.

Running DeepFace inference...


100%|██████████| 2677/2677 [09:12<00:00,  4.84it/s]


Saved DeepFace predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\deepface_gender_eval.csv
{'overall_accuracy': 0.7796040343668286, 'male_accuracy': 0.9377940407736539, 'female_accuracy': 0.38350785340314136, 'TP': 1794, 'TN': 293, 'FP': 471, 'FN': 119}
{1: {'N': 70, 'accuracy': 0.7142857142857143, 'male_accuracy': 0.8837209302325582, 'female_accuracy': 0.4444444444444444}, 2: {'N': 559, 'accuracy': 0.7799642218246869, 'male_accuracy': 0.9404466501240695, 'female_accuracy': 0.36538461538461536}, 3: {'N': 693, 'accuracy': 0.7691197691197691, 'male_accuracy': 0.907725321888412, 'female_accuracy': 0.4845814977973568}, 4: {'N': 485, 'accuracy': 0.8268041237113402, 'male_accuracy': 0.9578947368421052, 'female_accuracy': 0.3523809523809524}, 5: {'N': 349, 'accuracy': 0.7679083094555874, 'male_accuracy': 0.9437229437229437, 'female_accuracy': 0.423728813559322}, 6: {'N': 288, 'accuracy': 0.7534722222222222, 'male_accuracy': 0.9532710280373832, 'female_accu

## InsightFace

Link: https://github.com/deepinsight/insightface

In [7]:
import os
import cv2
import numpy as np
import pandas as pd
from tqdm import tqdm
from insightface.app import FaceAnalysis


# ============================================================================
# INSIGHTFACE MODEL (PADDING + BATCH PREDICTION)
# ============================================================================
class InsightFaceGenderClassifier:

    def __init__(self, det_size=(640, 640), pad_amount=1050, batch_size=32):
        self.pad_amount = pad_amount
        self.batch_size = batch_size
        try:
            self.app = FaceAnalysis(providers=['CUDAExecutionProvider', 'CPUExecutionProvider'])
        except:
            self.app = FaceAnalysis(providers=['CPUExecutionProvider'])
        
        self.app.prepare(ctx_id=0, det_size=det_size)
        print(f"✓ InsightFace initialized with batch_size={batch_size}.")

    def predict_batch(self, image_paths):
        """Process multiple images at once."""
        results = []
        
        for img_path in image_paths:
            img = cv2.imread(img_path)
            if img is None:
                results.append((None, None))
                continue

            # Pad image
            pad = self.pad_amount
            img_padded = cv2.copyMakeBorder(
                img,
                pad, pad, pad, pad,
                cv2.BORDER_CONSTANT,
                value=[0, 0, 0]
            )

            faces = self.app.get(img_padded)

            if len(faces) == 0:
                results.append((None, None))
                continue

            face = faces[0]
            gender = "Male" if face.gender == 1 else "Female"

            try:
                conf = float(face.gender_probability)
            except:
                conf = 1.0

            results.append((gender, conf))
        
        return results


# ============================================================================
# EVALUATION FUNCTION WITH BATCHING
# ============================================================================
def evaluate_insightface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image"):

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )
    
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = InsightFaceGenderClassifier(batch_size=batch_size)

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning InsightFace inference with batch_size={batch_size}...")

    # Prepare batches
    batch_paths = []
    batch_rows = []
    
    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row[image_type])

        if not os.path.exists(img_path):
            img_path = os.path.join(image_root, row[image_type])
            
        if os.path.exists(img_path):
            batch_paths.append(img_path)
            batch_rows.append(row)

    # Process in batches
    num_batches = (len(batch_paths) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(batch_paths), batch_size), total=num_batches):
        batch_end = min(i + batch_size, len(batch_paths))
        current_paths = batch_paths[i:batch_end]
        current_rows = batch_rows[i:batch_end]
        
        predictions = model.predict_batch(current_paths)
        
        for path, row, (pred_gender, conf) in zip(current_paths, current_rows, predictions):
            if pred_gender is None:
                continue
                
            all_true.append(row["gender"].lower())
            all_pred.append(pred_gender.lower())
            all_conf.append(conf)
            all_tone.append(int(row["mst_label"]))
            paths.append(path)

    # Save results
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved InsightFace predictions → {output_csv}")

    # Metrics (unchanged)
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0.0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df

### Casual Conversation v2

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\insightface_gender_eval.csv"

insightface_ccv2_faceonly_metrics, insightface_ccv2_faceonly_per_tone, insightface_ccv2_faceonly_result_df = evaluate_insightface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")

print(insightface_ccv2_faceonly_metrics)
print(insightface_ccv2_faceonly_per_tone)
print(insightface_ccv2_faceonly_result_df.head())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

  0%|          | 0/5559 [00:00<?, ?it/s]c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
100%|██████████| 5559/5559 [45:20<00:00,  2.04it/s]


Saved InsightFace predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\insightface_gender_eval.csv
{'overall_accuracy': 0.7708239881817475, 'male_accuracy': 0.7841286762400451, 'female_accuracy': 0.7626171297057373, 'TP': 50018, 'TN': 78863, 'FP': 24548, 'FN': 13770}
{1: {'N': 1135, 'accuracy': 0.7242290748898679, 'male_accuracy': 0.8047945205479452, 'female_accuracy': 0.6963226571767497}, 2: {'N': 12761, 'accuracy': 0.7824621894835828, 'male_accuracy': 0.7859810399310543, 'female_accuracy': 0.7811422413793103}, 3: {'N': 32350, 'accuracy': 0.7962287480680061, 'male_accuracy': 0.8159243993163768, 'female_accuracy': 0.7874838191313663}, 4: {'N': 36456, 'accuracy': 0.7763056835637481, 'male_accuracy': 0.7818406933553446, 'female_accuracy': 0.7735507990633088}, 5: {'N': 52279, 'accuracy': 0.7780753266129804, 'male_accuracy': 0.7979871209859696, 'female_accuracy': 0.7618799861255636}, 6: {'N': 20932, 'accuracy': 0.7376743741639595, 'male_accuracy': 0.7124090761046574, 'female

In [8]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Images"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\insightface_gender_eval.csv"

insightface_metrics, insightface_per_tone, insightface_result_df = evaluate_insightface(csv_path, image_root, output_csv, batch_size=32, image_type="original_image")

print(insightface_metrics)
print(insightface_per_tone)
print(insightface_result_df.head())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

  0%|          | 0/5559 [00:00<?, ?it/s]c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\insightface\utils\transform.py:68: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  P = np.linalg.lstsq(X_homo, Y)[0].T # Affine matrix. 3 x 4
100%|██████████| 5559/5559 [3:23:11<00:00,  2.19s/it]  


Saved InsightFace predictions → G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\insightface_gender_eval.csv
{'overall_accuracy': 0.8678668038894793, 'male_accuracy': 0.9707055304674213, 'female_accuracy': 0.803875594157414, 'TP': 66206, 'TN': 88112, 'FP': 21497, 'FN': 1998}
{1: {'N': 1190, 'accuracy': 0.7857142857142857, 'male_accuracy': 0.9835526315789473, 'female_accuracy': 0.7178329571106095}, 2: {'N': 13336, 'accuracy': 0.8811487702459508, 'male_accuracy': 0.9872386641325007, 'female_accuracy': 0.84067129389827}, 3: {'N': 33714, 'accuracy': 0.8880880346443614, 'male_accuracy': 0.9845504270223587, 'female_accuracy': 0.8449319538058644}, 4: {'N': 38257, 'accuracy': 0.8602085892777792, 'male_accuracy': 0.9785503548361375, 'female_accuracy': 0.8024965002333178}, 5: {'N': 55789, 'accuracy': 0.8783093441359408, 'male_accuracy': 0.9621687793705327, 'female_accuracy': 0.8086063152366573}, 6: {'N': 23073, 'accuracy': 0.8657738482208642, 'male_accuracy': 0.9597544338335607, 'f

### FACET

In [17]:
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\insightface_gender_eval.csv"

insightface_facet_faceonly_metrics, insightface_facet_faceonly_per_tone, insightface_facet_faceonly_result_df = evaluate_insightface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image")
print(insightface_facet_faceonly_metrics)
print(insightface_facet_faceonly_per_tone)
print(insightface_facet_faceonly_result_df.head())

Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider'], with options: {'CUDAExecutionProvider': {'device_id': '0', 'has_user_compute_stream': '0', 'cudnn_conv1d_pad_to_nc1d': '0', 'user_compute_stream': '0', 'gpu_external_alloc': '0', 'gpu_mem_limit': '18446744073709551615', 'enable_cuda_graph': '0', 'gpu_external_free': '0', 'gpu_external_empty_cache': '0', 'arena_extend_strategy': 'kNextPowerOfTwo', 'cudnn_conv_algo_search': 'EXHAUSTIVE', 'do_copy_in_default_stream': '1', 'cudnn_conv_use_max_workspace': '1', 'tunable_op_enable': '0', 'tunable_op_tuning_enable': '0', 'tunable_op_max_tuning_duration_ms': '0', 'enable_skip_layer_norm_strict_mode': '0', 'prefer_nhwc': '0', 'use_ep_level_unified_stream': '0', 'use_tf32': '1', 'sdpa_kernel': '0', 'fuse_conv_bias': '0'}, 'CPUExecutionProvider': {}}
find model: C:\Users\User/.insightface\models\buffalo_l\1k3d68.onnx landmark_3d_68 ['None', 3, 192, 192] 0.0 1.0
Applied providers: ['CUDAExecutionProvider', 'CPUExecutionProvider']

100%|██████████| 84/84 [01:25<00:00,  1.02s/it]

Saved InsightFace predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\insightface_gender_eval.csv
{'overall_accuracy': 0.7720323741007195, 'male_accuracy': 0.8211180124223603, 'female_accuracy': 0.6433224755700325, 'TP': 1322, 'TN': 395, 'FP': 219, 'FN': 288}
{1: {'N': 54, 'accuracy': 0.7037037037037037, 'male_accuracy': 0.6060606060606061, 'female_accuracy': 0.8571428571428571}, 2: {'N': 470, 'accuracy': 0.7808510638297872, 'male_accuracy': 0.8348082595870207, 'female_accuracy': 0.6412213740458015}, 3: {'N': 591, 'accuracy': 0.7783417935702199, 'male_accuracy': 0.8098765432098766, 'female_accuracy': 0.7096774193548387}, 4: {'N': 410, 'accuracy': 0.7829268292682927, 'male_accuracy': 0.8475609756097561, 'female_accuracy': 0.524390243902439}, 5: {'N': 287, 'accuracy': 0.7247386759581882, 'male_accuracy': 0.7575757575757576, 'female_accuracy': 0.651685393258427}, 6: {'N': 213, 'accuracy': 0.7887323943661971, 'male_accuracy': 0.85625, 'female_accuracy': 0

## FairFace

Link: https://github.com/dchen236/FairFace

In [1]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import pipeline


# ============================================================================
# FAIRFACE MODEL (HUGGINGFACE PIPELINE WITH BATCHING)
# ============================================================================
class FairFaceGenderClassifier:

    def __init__(self, batch_size=32):
        print("Loading FairFace model...")
        self.batch_size = batch_size
        self.classifier = pipeline(
            "image-classification",
            model="dima806/fairface_gender_image_detection",
            device=0 if torch.cuda.is_available() else -1,
            batch_size=batch_size  # Enable batching in pipeline
        )
        print(f"✓ FairFace loaded with batch_size={batch_size}.")

    def predict_batch(self, image_paths):
        """Process multiple images at once."""
        images = []
        valid_indices = []
        
        for idx, img_path in enumerate(image_paths):
            try:
                img = Image.open(img_path).convert("RGB")
                images.append(img)
                valid_indices.append(idx)
            except:
                pass
        
        if not images:
            return [(None, None)] * len(image_paths)
        
        try:
            # Process all images in batch
            results = self.classifier(images)
            
            # Map results back to original indices
            predictions = [(None, None)] * len(image_paths)
            for valid_idx, result in zip(valid_indices, results):
                label = result[0]["label"]
                conf = float(result[0]["score"])
                predictions[valid_idx] = (label, conf)
            
            return predictions
        except:
            return [(None, None)] * len(image_paths)


# ============================================================================
# EVALUATION FUNCTION WITH BATCHING
# ============================================================================
def evaluate_fairface(csv_path, image_root, output_csv, batch_size=32, image_type="cropped_image"):

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )
    
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]                # removes NaN
    df = df[df["gender"].astype(str) != ""]      # removes empty string
    df = df[df["gender"].astype(str) != "nan"]   # extra safety

    model = FairFaceGenderClassifier(batch_size=batch_size)

    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning FairFace inference with batch_size={batch_size}...")

    # Prepare batches
    batch_paths = []
    batch_rows = []
    
    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row[image_type])

        if not os.path.exists(img_path):
            img_path = os.path.join(image_root, row[image_type])
            
        if os.path.exists(img_path):
            batch_paths.append(img_path)
            batch_rows.append(row)

    # Process in batches
    num_batches = (len(batch_paths) + batch_size - 1) // batch_size
    
    for i in tqdm(range(0, len(batch_paths), batch_size), total=num_batches):
        batch_end = min(i + batch_size, len(batch_paths))
        current_paths = batch_paths[i:batch_end]
        current_rows = batch_rows[i:batch_end]
        
        predictions = model.predict_batch(current_paths)
        
        for path, row, (pred_gender, conf) in zip(current_paths, current_rows, predictions):
            if pred_gender is None:
                continue
                
            all_true.append(row["gender"].lower())
            all_pred.append(pred_gender.lower())
            all_conf.append(conf)
            all_tone.append(int(row["mst_label"]))
            paths.append(path)

    # Save CSV
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })
    out_df.to_csv(output_csv, index=False)
    print(f"Saved FairFace predictions → {output_csv}")

    # Metrics (unchanged)
    def acc(y, yp): return np.mean(np.array(y) == np.array(yp))
    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0: return 0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t=="male")   and (p=="male")   for t,p in zip(y,yp))
        TN = sum((t=="female") and (p=="female") for t,p in zip(y,yp))
        FP = sum((t=="female") and (p=="male")   for t,p in zip(y,yp))
        FN = sum((t=="male")   and (p=="female") for t,p in zip(y,yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female")
    }
    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP":TP, "TN":TN, "FP":FP, "FN":FN})

    # Per-tone breakdown
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]
        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female")
        }

    return metrics, per_tone, out_df

c:\MastersRepos\ARI5902-Research-Topics-in-AI\LAION-5B Testing\DatasetAnnotation\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import pipeline
from torch.utils.data import Dataset


# ============================================================================
# DATASET FOR HUGGINGFACE PIPELINE STREAMING
# ============================================================================
class ImagePathDataset(Dataset):
    """
    Lightweight dataset that yields image paths.
    Hugging Face pipelines will handle image loading internally.
    """
    def __init__(self, image_paths):
        self.image_paths = image_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        return self.image_paths[idx]


# ============================================================================
# FAIRFACE MODEL (HUGGINGFACE PIPELINE WITH TRUE GPU BATCHING)
# ============================================================================
class FairFaceGenderClassifier:

    def __init__(self, batch_size=32):
        print("Loading FairFace model...")
        self.batch_size = batch_size

        self.classifier = pipeline(
            "image-classification",
            model="dima806/fairface_gender_image_detection",
            device=0 if torch.cuda.is_available() else -1,
            batch_size=batch_size
        )

        print(
            f"✓ FairFace loaded | device="
            f"{'cuda' if torch.cuda.is_available() else 'cpu'} | "
            f"batch_size={batch_size}"
        )


# ============================================================================
# EVALUATION FUNCTION (DATASET-BASED PIPELINE STREAMING)
# ============================================================================
def evaluate_fairface(
    csv_path,
    image_root,
    output_csv,
    batch_size=32,
    image_type="cropped_image"
):
    """
    Evaluate FairFace gender classification using GPU-efficient batching.

    Args:
        csv_path: CSV with metadata
        image_root: Root directory of images
        output_csv: Where predictions will be saved
        batch_size: HF pipeline batch size
        image_type: 'cropped_image' or 'original_image'
    """

    allowed_image_types = {"cropped_image", "original_image"}
    if image_type not in allowed_image_types:
        raise ValueError(
            f"Invalid image_type='{image_type}'. "
            f"Must be one of {allowed_image_types}."
        )

    # ------------------------------------------------------------------
    # LOAD + CLEAN METADATA
    # ------------------------------------------------------------------
    df = pd.read_csv(csv_path, dtype={"subject_id": str})
    df["subject_id"] = df["subject_id"].str.zfill(4)

    df = df[df["gender"].notna()]
    df = df[df["gender"].astype(str) != ""]
    df = df[df["gender"].astype(str) != "nan"]

    # ------------------------------------------------------------------
    # COLLECT IMAGE PATHS + METADATA
    # ------------------------------------------------------------------
    image_paths = []
    rows = []

    for _, row in df.iterrows():
        img_path = os.path.join(image_root, row["subject_id"], row[image_type])
        if not os.path.exists(img_path):
            img_path = os.path.join(image_root, row[image_type])

        if os.path.exists(img_path):
            image_paths.append(img_path)
            rows.append(row)

    print(f"Found {len(image_paths)} images to process.")

    # ------------------------------------------------------------------
    # MODEL
    # ------------------------------------------------------------------
    model = FairFaceGenderClassifier(batch_size=batch_size)

    dataset = ImagePathDataset(image_paths)

    # ------------------------------------------------------------------
    # INFERENCE (STREAMED, GPU-EFFICIENT)
    # ------------------------------------------------------------------
    all_true, all_pred, all_conf, all_tone, paths = [], [], [], [], []

    print(f"\nRunning FairFace inference (streaming, batch_size={batch_size})...")

    for output, row, path in tqdm(
        zip(
            model.classifier(dataset, batch_size=batch_size),
            rows,
            image_paths
        ),
        total=len(dataset),
        desc="FairFace inference"
    ):
        # HF returns list of predictions per image
        pred = output[0]
        pred_gender = pred["label"].lower()
        conf = float(pred["score"])

        all_true.append(row["gender"].lower())
        all_pred.append(pred_gender)
        all_conf.append(conf)
        all_tone.append(int(row["mst_label"]))
        paths.append(path)

    # ------------------------------------------------------------------
    # SAVE CSV
    # ------------------------------------------------------------------
    out_df = pd.DataFrame({
        "image_path": paths,
        "true_gender": all_true,
        "pred_gender": all_pred,
        "confidence": all_conf,
        "mst_label": all_tone,
    })

    out_df.to_csv(output_csv, index=False)
    print(f"\nSaved FairFace predictions → {output_csv}")

    # ------------------------------------------------------------------
    # METRICS
    # ------------------------------------------------------------------
    def acc(y, yp):
        return np.mean(np.array(y) == np.array(yp))

    def acc_class(y, yp, lbl):
        idx = [i for i, g in enumerate(y) if g == lbl]
        if len(idx) == 0:
            return 0.0
        return np.mean([yp[j] == lbl for j in idx])

    def confmat(y, yp):
        TP = sum((t == "male")   and (p == "male")   for t, p in zip(y, yp))
        TN = sum((t == "female") and (p == "female") for t, p in zip(y, yp))
        FP = sum((t == "female") and (p == "male")   for t, p in zip(y, yp))
        FN = sum((t == "male")   and (p == "female") for t, p in zip(y, yp))
        return TP, TN, FP, FN

    metrics = {
        "overall_accuracy": acc(all_true, all_pred),
        "male_accuracy": acc_class(all_true, all_pred, "male"),
        "female_accuracy": acc_class(all_true, all_pred, "female"),
    }

    TP, TN, FP, FN = confmat(all_true, all_pred)
    metrics.update({"TP": TP, "TN": TN, "FP": FP, "FN": FN})

    # ------------------------------------------------------------------
    # PER-TONE METRICS
    # ------------------------------------------------------------------
    per_tone = {}
    for tone in sorted(set(all_tone)):
        idx = [i for i, t in enumerate(all_tone) if t == tone]
        yt = [all_true[j] for j in idx]
        yp = [all_pred[j] for j in idx]

        per_tone[tone] = {
            "N": len(idx),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female"),
        }

    return metrics, per_tone, out_df


### Casual Conversation v2

In [ ]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\fairface_gender_eval.csv"

fairface_ccv2_faceonly_metrics, fairface_ccv2_faceonly_per_tone, fairface_ccv2_faceonly_result_df = evaluate_fairface(csv_path, image_root, output_csv, image_type="cropped_image")

print(fairface_ccv2_faceonly_metrics)
print(fairface_ccv2_faceonly_per_tone)
print(fairface_ccv2_faceonly_result_df.head())

Loading FairFace model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


✓ FairFace loaded with batch_size=32.

Running FairFace inference with batch_size=32...


100%|██████████| 5559/5559 [24:30<00:00,  3.78it/s]  


Saved FairFace predictions → G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\fairface_gender_eval.csv
{'overall_accuracy': 0.8006330715213394, 'male_accuracy': 0.983216557708657, 'female_accuracy': 0.6870240147390119, 'TP': 67077, 'TN': 75326, 'FP': 34315, 'FN': 1145}
{1: {'N': 1190, 'accuracy': 0.6840336134453782, 'male_accuracy': 1.0, 'female_accuracy': 0.5756207674943566}, 2: {'N': 13337, 'accuracy': 0.7647896828372198, 'male_accuracy': 0.9915829486831388, 'female_accuracy': 0.6782680754091568}, 3: {'N': 33732, 'accuracy': 0.7890726906201826, 'male_accuracy': 0.9904085938998657, 'female_accuracy': 0.6990045481850168}, 4: {'N': 38273, 'accuracy': 0.7833982180649544, 'male_accuracy': 0.9857336415079302, 'female_accuracy': 0.6847158516675736}, 5: {'N': 55797, 'accuracy': 0.8290409878667312, 'male_accuracy': 0.9827858496525584, 'female_accuracy': 0.701237323180938}, 6: {'N': 23074, 'accuracy': 0.8106960214960561, 'male_accuracy': 0.9690118885207561, 'female_accuracy': 0.6838901030

In [3]:
csv_path = r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\annotations.csv"
image_root = r"G:\Thesis\CasualConversationv2_Dataset\Images"
output_csv = r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\fairface_gender_eval.csv"

fairface_ccv2_metrics, fairface_ccv2_per_tone, fairface_ccv2_result_df = evaluate_fairface(csv_path, image_root, output_csv, image_type="original_image")

print(fairface_ccv2_metrics)
print(fairface_ccv2_per_tone)
print(fairface_ccv2_result_df.head())

Found 177863 images to process.
Loading FairFace model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


✓ FairFace loaded | device=cuda | batch_size=32

Running FairFace inference (streaming, batch_size=32)...


FairFace inference: 100%|██████████| 177863/177863 [2:01:36<00:00, 24.37it/s]  



Saved FairFace predictions → G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\fairface_gender_eval.csv
{'overall_accuracy': 0.9409939110438933, 'male_accuracy': 0.94613174635748, 'female_accuracy': 0.9377969920011674, 'TP': 64547, 'TN': 102821, 'FP': 6820, 'FN': 3675}
{1: {'N': 1190, 'accuracy': 0.9705882352941176, 'male_accuracy': 1.0, 'female_accuracy': 0.9604966139954854}, 2: {'N': 13337, 'accuracy': 0.9370922996176052, 'male_accuracy': 0.9446103719793646, 'female_accuracy': 0.934224155790346}, 3: {'N': 33732, 'accuracy': 0.9435550812285071, 'male_accuracy': 0.9397659696911568, 'female_accuracy': 0.9452501501759204}, 4: {'N': 38273, 'accuracy': 0.9409505395448489, 'male_accuracy': 0.9352036343349007, 'female_accuracy': 0.9437534012283293}, 5: {'N': 55797, 'accuracy': 0.9405882036668638, 'male_accuracy': 0.9518319646241314, 'female_accuracy': 0.9312415898125964}, 6: {'N': 23074, 'accuracy': 0.9429661090404785, 'male_accuracy': 0.9374390956928474, 'female_accuracy': 0.9

### FACET

In [2]:
csv_path = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\annotations.csv"
image_root = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label"
output_csv = r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\fairface_gender_eval.csv"

fairface_facet_faceonly_metrics, fairface_facet_faceonly_per_tone, fairface_facet_faceonly_result_df = evaluate_fairface(csv_path, image_root, output_csv, image_type="cropped_image")
print(fairface_facet_faceonly_metrics)
print(fairface_facet_faceonly_per_tone)
print(fairface_facet_faceonly_result_df.head())

Loading FairFace model...


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Device set to use cuda:0


✓ FairFace loaded with batch_size=32.

Running FairFace inference with batch_size=32...


100%|██████████| 84/84 [00:24<00:00,  3.48it/s]

Saved FairFace predictions → G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\fairface_gender_eval.csv
{'overall_accuracy': 0.8341426970489354, 'male_accuracy': 0.9064296915838996, 'female_accuracy': 0.6531413612565445, 'TP': 1734, 'TN': 499, 'FP': 265, 'FN': 179}
{1: {'N': 70, 'accuracy': 0.7285714285714285, 'male_accuracy': 0.7209302325581395, 'female_accuracy': 0.7407407407407407}, 2: {'N': 559, 'accuracy': 0.8568872987477638, 'male_accuracy': 0.9280397022332506, 'female_accuracy': 0.6730769230769231}, 3: {'N': 693, 'accuracy': 0.8124098124098124, 'male_accuracy': 0.8948497854077253, 'female_accuracy': 0.6431718061674009}, 4: {'N': 485, 'accuracy': 0.8412371134020619, 'male_accuracy': 0.9184210526315789, 'female_accuracy': 0.5619047619047619}, 5: {'N': 349, 'accuracy': 0.8424068767908309, 'male_accuracy': 0.8961038961038961, 'female_accuracy': 0.7372881355932204}, 6: {'N': 288, 'accuracy': 0.8472222222222222, 'male_accuracy': 0.9065420560747663, 'female_accur

# Result Visualisation

In [5]:
import pandas as pd
import numpy as np

def evaluate_gender_csv(csv_path):
    """
    Evaluate gender classification metrics from a CSV file.

    Expected columns:
        - image_path
        - true_gender  (male / female)
        - pred_gender  (male / female)
        - confidence
        - mst_label    (int)

    Returns:
        metrics: dict
        per_tone: dict
        df: cleaned DataFrame
    """

    # ------------------------------------------------------------------
    # Load + clean
    # ------------------------------------------------------------------
    df = pd.read_csv(csv_path)

    required_cols = {
        "image_path", "true_gender", "pred_gender", "confidence", "mst_label"
    }
    missing = required_cols - set(df.columns)
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    df = df.dropna(subset=["true_gender", "pred_gender", "mst_label"])
    df["true_gender"] = df["true_gender"].str.lower().str.strip()
    df["pred_gender"] = df["pred_gender"].str.lower().str.strip()
    df["mst_label"] = df["mst_label"].astype(int)

    # ------------------------------------------------------------------
    # Metric helpers
    # ------------------------------------------------------------------
    def acc(y, yp):
        return np.mean(np.array(y) == np.array(yp))

    def acc_class(y, yp, cls):
        idx = [i for i, v in enumerate(y) if v == cls]
        if len(idx) == 0:
            return 0.0
        return np.mean([yp[i] == cls for i in idx])

    def confmat(y, yp):
        TP = sum((t == "male")   and (p == "male")   for t, p in zip(y, yp))
        TN = sum((t == "female") and (p == "female") for t, p in zip(y, yp))
        FP = sum((t == "female") and (p == "male")   for t, p in zip(y, yp))
        FN = sum((t == "male")   and (p == "female") for t, p in zip(y, yp))
        return TP, TN, FP, FN

    # ------------------------------------------------------------------
    # Global metrics
    # ------------------------------------------------------------------
    y_true = df["true_gender"].tolist()
    y_pred = df["pred_gender"].tolist()

    metrics = {
        "overall_accuracy": acc(y_true, y_pred),
        "male_accuracy": acc_class(y_true, y_pred, "male"),
        "female_accuracy": acc_class(y_true, y_pred, "female"),
    }

    TP, TN, FP, FN = confmat(y_true, y_pred)
    metrics.update({
        "TP": TP,
        "TN": TN,
        "FP": FP,
        "FN": FN,
    })

    # ------------------------------------------------------------------
    # Per-tone metrics
    # ------------------------------------------------------------------
    per_tone = {}

    for tone in sorted(df["mst_label"].unique()):
        df_t = df[df["mst_label"] == tone]

        yt = df_t["true_gender"].tolist()
        yp = df_t["pred_gender"].tolist()

        per_tone[tone] = {
            "N": len(df_t),
            "accuracy": acc(yt, yp),
            "male_accuracy": acc_class(yt, yp, "male"),
            "female_accuracy": acc_class(yt, yp, "female"),
        }

    return metrics, per_tone, df

def evaluate_multiple_gender_csvs(models):
    """
    Evaluate multiple gender CSV outputs and produce comparison tables.

    Args:
        models: list of dicts, each with:
            {
                "name": str,
                "csv_path": str
            }

    Returns:
        overall_df: DataFrame with global metrics per model
        per_tone_df: DataFrame with per-tone accuracy per model
    """

    overall_rows = []
    per_tone_rows = []

    for m in models:
        name = m["name"]
        csv_path = m["csv_path"]

        metrics, per_tone, _ = evaluate_gender_csv(csv_path)

        # -----------------------------
        # Overall metrics table
        # -----------------------------
        overall_rows.append({
            "model": name,
            "overall_accuracy": metrics["overall_accuracy"],
            "male_accuracy": metrics["male_accuracy"],
            "female_accuracy": metrics["female_accuracy"],
            "TP": metrics["TP"],
            "TN": metrics["TN"],
            "FP": metrics["FP"],
            "FN": metrics["FN"],
        })

        # -----------------------------
        # Per-tone metrics table
        # -----------------------------
        for tone, stats in per_tone.items():
            per_tone_rows.append({
                "model": name,
                "mst_label": tone,
                "N": stats["N"],
                "accuracy": stats["accuracy"],
                "male_accuracy": stats["male_accuracy"],
                "female_accuracy": stats["female_accuracy"],
            })

    overall_df = pd.DataFrame(overall_rows).set_index("model")
    per_tone_df = pd.DataFrame(per_tone_rows).set_index(["mst_label", "model"])

    return overall_df, per_tone_df

### CCv2 Metric Visualisation

In [9]:
models = [
    {
        "name": "CCv2 HF-Gender",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\hf_gender_eval.csv",
    },
    {
        "name": "CCv2 Realistic",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\realistic_gender_eval.csv",
    },
    # Deep Face was not executed as its executaion time was too much given it doesn't support true batching
    # {
    #     "name": "CCv2 DeepFace",
    #     "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\deepface_gender_eval.csv",
    # },
    {
        "name": "CCv2 InsightFace",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\insightface_gender_eval.csv",
    },
    {
        "name": "CCv2 FairFace",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Images\GenderEvaluation\fairface_gender_eval.csv",
    }
]

overall_df, per_tone_df = evaluate_multiple_gender_csvs(models)
print("\n===== OVERALL METRICS COMPARISON =====")
display(overall_df.round(4))


===== OVERALL METRICS COMPARISON =====


,overall_accuracy,male_accuracy,female_accuracy,TP,TN,FP,FN
model,,,,,,,
CCv2 HF-Gender,0.8313,0.9870,0.7344,67332,80522,29119,890
CCv2 Realistic,0.8159,0.9889,0.7083,67467,77655,31986,755
CCv2 InsightFace,0.8679,0.9707,0.8039,66206,88112,21497,1998
CCv2 FairFace,0.9410,0.9461,0.9378,64547,102821,6820,3675


### CCv2 Face Only Metric Visualisation

In [ ]:
models = [
    {
        "name": "CCv2 FaceOnly HF-Gender",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\hf_gender_eval.csv",
    },
    {
        "name": "CCv2 FaceOnly Realistic",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\realistic_gender_eval.csv",
    },
    # Deep Face was not executed as its executaion time was too much given it doesn't support true batching
    # {
    #     "name": "CCv2 FaceOnly DeepFace",
    #     "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\deepface_gender_eval.csv",
    # },
    {
        "name": "CCv2 FaceOnly InsightFace",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\insightface_gender_eval.csv",
    },
    {
        "name": "CCv2 FaceOnly FairFace",
        "csv_path": r"G:\Thesis\CasualConversationv2_Dataset\Segmented_CCV2\GenderEvaluation\fairface_gender_eval.csv",
    }
]

overall_df, per_tone_df = evaluate_multiple_gender_csvs(models)
print("\n===== OVERALL METRICS COMPARISON =====")
display(overall_df.round(4))


===== OVERALL METRICS COMPARISON =====


,overall_accuracy,male_accuracy,female_accuracy,TP,TN,FP,FN
model,,,,,,,
CCv2 FaceOnly HF-Gender,0.7152,0.9626,0.5613,65672,61543,48098,2550
CCv2 FaceOnly Realistic,0.8858,0.9242,0.8619,63054,94499,15142,5168
CCv2 FaceOnly InsightFace,0.7708,0.7841,0.7626,50018,78863,24548,13770
CCv2 FaceOnly FairFace,0.8006,0.9832,0.6870,67077,75326,34315,1145


### FACET Metric Visualisation

In [11]:
models = [
    {
        "name": "FACET HF-Gender",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\hf_gender_eval.csv",
    },
    {
        "name": "FACET Realistic",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\realistic_gender_eval.csv",
    },
    {
        "name": "FACET DeepFace",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\deepface_gender_eval.csv",
    },
    {
        "name": "FACET InsightFace",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\insightface_gender_eval.csv",
    },
    {
        "name": "FACET FairFace",
        "csv_path": r"G:\Thesis\FACET_Dataset\Segmented_FACET_0.2_fixed_label\GenderEvaluation\fairface_gender_eval.csv",
    }
]

overall_df, per_tone_df = evaluate_multiple_gender_csvs(models)
print("\n===== OVERALL METRICS COMPARISON =====")
display(overall_df.round(4))

# print("\n===== PER-TONE ACCURACY COMPARISON =====")
# display(per_tone_df.round(4))


===== OVERALL METRICS COMPARISON =====


,overall_accuracy,male_accuracy,female_accuracy,TP,TN,FP,FN
model,,,,,,,
FACET HF-Gender,0.7889,0.8766,0.5694,1677,435,329,236
FACET Realistic,0.8233,0.8745,0.6950,1673,531,233,240
FACET DeepFace,0.7796,0.9378,0.3835,1794,293,471,119
FACET InsightFace,0.7720,0.8211,0.6433,1322,395,219,288
FACET FairFace,0.8341,0.9064,0.6531,1734,499,265,179


### Result

The best performing model appears to be the **Realistic Gender Classifier** as it's the most accurate and a relatively fast model, producign the following metrics on CCv2 FaceOnly

- Overall Accuracy: 0.886
    - Male Accuracy: 0.924
    - Female Accuracy': 0.862

- Confusion Matrix:
    - TP: 63054
    - TN: 94499
    - FP: 15142
    - FN: 5168

- Per SkinTone Accuracy:

    - **Skin Tone 1**  
        - N = 1,190  
        - Accuracy: **0.756**  
            - Male Accuracy: 0.977  
            - Female Accuracy: 0.681  

    - **Skin Tone 2**  
        - N = 13,337  
        - Accuracy: **0.870**  
            - Male Accuracy: 0.968  
            - Female Accuracy: 0.833  

    - **Skin Tone 3**  
        - N = 33,732  
        - Accuracy: **0.893**  
            - Male Accuracy: 0.946  
            - Female Accuracy: 0.870  

    - **Skin Tone 4**  
        - N = 38,273  
        - Accuracy: **0.893**  
            - Male Accuracy: 0.939  
            - Female Accuracy: 0.871  

    - **Skin Tone 5**  
        - N = 55,797  
        - Accuracy: **0.906**  
            - Male Accuracy: 0.918  
            - Female Accuracy: 0.895  

    - **Skin Tone 6**  
        - N = 23,074  
        - Accuracy: **0.879**  
            - Male Accuracy: 0.884  
            - Female Accuracy: 0.875  

    - **Skin Tone 7**  
        - N = 6,438  
        - Accuracy: **0.826**  
            - Male Accuracy: 0.906  
            - Female Accuracy: 0.761  

    - **Skin Tone 8**  
        - N = 4,183  
        - Accuracy: **0.803**  
            - Male Accuracy: 0.925  
            - Female Accuracy: 0.680  

    - **Skin Tone 9**  
        - N = 1,671  
        - Accuracy: **0.662**  
            - Male Accuracy: 0.948  
            - Female Accuracy: 0.488  

    - **Skin Tone 10**  
        - N = 168  
        - Accuracy: **0.726**  
            - Male Accuracy: 1.000  
            - Female Accuracy: 0.570  
